In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nusratjahannawal/banglapsg-v2/BanglaPSG-Grievance-Dataset_V2.xlsx


In [2]:
!pip install -q transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding 
)

In [4]:
path = "/kaggle/input/datasets/nusratjahannawal/banglapsg-v2/BanglaPSG-Grievance-Dataset_V2.xlsx"

df = pd.read_excel(path)

df.head()

,No.,Grievance Text,Category,Severity
0,1,সরকারি হাসপাতাল গুলো এত অপরিষ্কার আর নোংরা।,Healthcare,High
1,2,মেডিসিন বিশেষজ্ঞ ডাঃ বসে না এসিস্ট্যান্ট দিয়ে ...,Healthcare,High
2,3,রূপগঞ্জে মানুষ মারার কারখানা ইউ এস-বাংলা হাসপাতাল,Healthcare,Low
3,4,সরকারি হাসপাতালগুলোর সমস্ত টেস্টের মেশিন গুলো ...,Healthcare,Medium
4,5,সামান্য জ্বর আসলে এক হাজার নয়শত টাকার টেস্ট দয়,Healthcare,Low


In [5]:
data = df[["Grievance Text", "Category"]].copy()

data.columns = ["text", "label"]


label_encoder = LabelEncoder()

data["label"] = label_encoder.fit_transform(data["label"])


print(label_encoder.classes_)

num_labels = len(label_encoder.classes_)

print("Classes:", num_labels)

['Electricity' 'Food' 'Healthcare' 'Road & Transport' 'Waste' 'Water']
Classes: 6


In [6]:
train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    random_state=42,
    stratify=data["label"]
)


valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)


print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(4273, 2)
(916, 2)
(916, 2)


In [7]:
train_dataset = Dataset.from_pandas(train_df)

valid_dataset = Dataset.from_pandas(valid_df)

test_dataset = Dataset.from_pandas(test_df)

In [8]:
model_name = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )


train_dataset = train_dataset.map(tokenize, batched=True)

valid_dataset = valid_dataset.map(tokenize, batched=True)

test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/4273 [00:00<?, ? examples/s]

Map:   0%|          | 0/916 [00:00<?, ? examples/s]

Map:   0%|          | 0/916 [00:00<?, ? examples/s]

In [10]:
for dataset in [train_dataset, valid_dataset, test_dataset]:

    dataset.set_format(
        "torch",
        columns=[
            "input_ids",
            "attention_mask",
            "label"
        ]
    )

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)


    return {
        "accuracy": accuracy_score(labels, predictions),

        "precision": precision_score(
            labels,
            predictions,
            average="macro"
        ),

        "recall": recall_score(
            labels,
            predictions,
            average="macro"
        ),

        "f1": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }

In [13]:
training_args = TrainingArguments(

    output_dir="./mbert_category",

    num_train_epochs=3,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    fp16=True,

    logging_steps=50,

    report_to="none"
)

In [14]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [15]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.290015,1.047053,0.837336,0.853530,0.838328,0.841505
2,0.748023,0.661231,0.906114,0.906049,0.905938,0.905919
3,0.458593,0.672563,0.913755,0.915432,0.913416,0.913638


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=804, training_loss=1.1421611535905012, metrics={'train_runtime': 307.7608, 'train_samples_per_second': 41.652, 'train_steps_per_second': 2.612, 'total_flos': 843235437860352.0, 'train_loss': 1.1421611535905012, 'epoch': 3.0})

In [16]:
result = trainer.evaluate(test_dataset)

result

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.6910259127616882,
 'eval_accuracy': 0.9093886462882096,
 'eval_precision': 0.9124707219006698,
 'eval_recall': 0.9088989033981316,
 'eval_f1': 0.9098952162915156,
 'eval_runtime': 6.5809,
 'eval_samples_per_second': 139.19,
 'eval_steps_per_second': 8.813,
 'epoch': 3.0}

In [17]:
pred = trainer.predict(test_dataset)

y_pred = np.argmax(pred.predictions, axis=1)

y_true = test_df["label"].values


print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


                  precision    recall  f1-score   support

     Electricity       0.96      0.93      0.94       151
            Food       0.91      0.95      0.93       156
      Healthcare       0.97      0.90      0.94       152
Road & Transport       0.84      0.92      0.88       159
           Waste       0.89      0.88      0.88       153
           Water       0.91      0.88      0.89       145

        accuracy                           0.91       916
       macro avg       0.91      0.91      0.91       916
    weighted avg       0.91      0.91      0.91       916



In [18]:
model_save_path = "./mBert_Category_Model"

trainer.save_model(model_save_path)

tokenizer.save_pretrained(model_save_path)

print("Saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved
